# 03. Inference

Dự đoán văn bản mới bằng model đã train, và kiểm tra rằng gộp LoRA vào trọng số gốc không làm đổi kết quả.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Đổi thành địa chỉ repo của bạn. Repo private thì thêm secret GITHUB_TOKEN trên Colab.
REPO_URL = "https://github.com/TEN-TAI-KHOAN/vihsd-lora.git"

def github_token():
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

def find_repo():
    places = [Path.cwd(), *Path.cwd().parents, Path("/content/vihsd-lora"), Path("/kaggle/working/vihsd-lora")]
    places += list(Path("/kaggle/input").glob("*")) + list(Path("/kaggle/input").glob("*/vihsd-lora"))
    return next((place for place in places if (place / "src" / "vihsd_lora").is_dir()), None)

repo = find_repo()
if repo is None:
    if "TEN-TAI-KHOAN" in REPO_URL:
        raise ValueError("Hãy sửa REPO_URL thành địa chỉ repo GitHub của bạn rồi chạy lại ô này.")
    repo = Path("/content" if Path("/content").exists() else "/kaggle/working") / "vihsd-lora"
    token = github_token()
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url, str(repo)], check=True)
elif (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=False)
if not os.environ.get("VIHSD_SKIP_INSTALL"):
    # Colab cài sẵn torchao bản cũ, peft sẽ báo lỗi khi gắn LoRA. Repo không dùng torchao nên gỡ đi.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
sys.path.insert(0, str(repo / "src"))
print("Repo:", repo)

In [ ]:
from vihsd_lora.config import Settings
from vihsd_lora.env import default_output_root, describe_gpu, detect_platform, free_space_gb, load_secrets

print("Nền tảng:", detect_platform())
print("GPU:", describe_gpu())
print("Secret đã có:", load_secrets())

settings = Settings.from_env(output_root=default_output_root())
free_gb = free_space_gb(settings.output_root)
print("Kết quả lưu tại:", settings.output_root)
print(f"Dung lượng còn trống: {free_gb} GB")
if free_gb < 6:
    print("Cảnh báo: còn dưới 6GB. Lượt full fine-tune cần khoảng 3GB, hãy dọn Drive và làm trống thùng rác.")

In [ ]:
from pathlib import Path
import pandas as pd
from vihsd_lora.evaluate import select_for_test
from vihsd_lora.inference import Predictor

SLUG = select_for_test(settings)["lora"]
predictor = Predictor(Path(settings.output_root) / "runs" / SLUG)
print("Dùng lượt:", SLUG)

## Dự đoán

Thay danh sách dưới bằng câu cần thử.

In [ ]:
TEXTS = [
    "Hôm nay thời tiết đẹp quá, cả nhà đi chơi vui nhé.",
    "Bài viết rất hữu ích, cảm ơn bạn đã chia sẻ.",
]
before = pd.DataFrame(predictor.predict(TEXTS))
display(before)

## Gộp LoRA vào trọng số gốc

Sau khi gộp, kết quả phải gần như giữ nguyên. Đây là lý do LoRA không làm tăng độ trễ khi triển khai.

In [ ]:
latency_adapter = predictor.latency_ms(TEXTS, batch_size=1)
predictor.merge_lora()
latency_merged = predictor.latency_ms(TEXTS, batch_size=1)
after = pd.DataFrame(predictor.predict(TEXTS))

print(f"Độ trễ mỗi câu: chưa gộp {latency_adapter} ms, đã gộp {latency_merged} ms")
print("Nhãn giữ nguyên:", (before["label"] == after["label"]).all())
display(after)